<a href="https://colab.research.google.com/github/gosainkrishika-collab/AI-Household-Assistant/blob/main/House_AI_asssitant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# import the libraries

In [ ]:
!pip install -qU langgraph langchain-groq pandas

# Import the api

In [ ]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key:")


# Create shared emmory

In [ ]:
from typing import TypedDict, Literal, Optional

class HouseState(TypedDict):
   name:str
   user_query: str
   intent: Literal['food_query', 'appliance_query', 'energy_query', 'unknown']
   food_item: str
   expiry_status: Literal['expired','fresh','unknown']
   expiry_date: Optional[str] # Made optional(field has a str value or nothing)
   safety_status: Literal['safe', 'unsafe', 'unknown']
   appliance_problem: str
   energy_analysis: str
   recommendation: str
   final_response: str

# create intake node

In [ ]:
def intake_node(state:HouseState)-> HouseState:
  print("=== House Intake Form ===")
  name = input("Patient name: ")
  user_query = input("Describe your problem: ")
  return{
      "name": name,
      "user_query": user_query,
      "intent": "unknown",
      "food_item": "", #implies no specific food item has been identified
      "expiry_status": "unknown", #implies that the state is currently undetermined
      "expiry_date": "",
      "safety_status": "unknown",
      "appliance_problem": "",
      "energy_analysis": "",
      "recommendation": "",
  }


#Create router node ( intent node )

In [ ]:
from langchain_groq import ChatGroq # enable node to chat with groq api
from langchain_core.messages import SystemMessage, HumanMessage #system message:system prompt, Human Message: Human prompt

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0) #temp high, llm mre creative if temp =1, LLM creativity high

INTENT_SYSTEM_PROMPT = """You are an intent router for a home AI assistant. Your task is to analyze the user's query and classify its intent into one of the following categories:

- `food_query`: The user is asking about food safety, expiration, recipes, or anything related to food items.
- `appliance_query`: The user is asking about an appliance, its status, troubleshooting, or general information.
- `energy_query`: The user is asking about energy consumption, smart home energy management, or related topics.
- `unknown`: The user's query does not fit into any of the above categories.

Respond with ONLY the name of the intent category, e.g., 'food_query', 'appliance_query', 'energy_query', or 'unknown'. Do not provide any other text, explanations, or punctuation.
"""

# Hard safety-net keywords — these always override the LLM, since misrouting
# a genuine crisis is unacceptable. Good example of "guardrail before AI".
FOOD_QUERY_KEYWORDS = [""]
CRISIS_KEYWORDS = ["suicide", "kill myself", "self-harm", "want to die"]